# Arquitectura de despliegue real del sistema

Este documento describe cómo está desplegado el sistema actualmente, que difiere en algunos puntos de lo que quedó en el informe de avance.

---

## Infraestructura

EC2 de AWS Academy, instancia **t3.small** (2 vCPU, 2 GB RAM), Ubuntu 24.04 LTS. Los puertos 8000 y 8501 están abiertos en el security group. La IP pública cambia cada vez que se prende la instancia.

---

## Dos contenedores con Docker Compose

El sistema corre como dos contenedores orquestados con Docker Compose desde `docker/docker-compose.yml`.

**Contenedor API — puerto 8000**

FastAPI servida con uvicorn. Expone cuatro endpoints:

- `GET /health` — verificación de que el servidor está vivo
- `GET /models/{task}` — lista los modelos disponibles para task1 o task2
- `POST /analyze` — recibe texto, lo parte en párrafos, corre T1 + T2 sobre cada uno y devuelve el JSON completo con segmentos, etiquetas, confianzas y resumen
- `POST /compare` — fija la segmentación de T1 y corre los tres modelos de T2 sobre el mismo input — esto es la comparación de arquitecturas que pide el enunciado

La imagen se construye desde `docker/Dockerfile` con `python:3.11-slim`. Los pesos de los modelos **no van dentro de la imagen** — se montan como volumen desde la carpeta `models/` del repo en runtime. Incluirlos significaría reconstruir y subir gigas cada vez que cambia una línea de código.

**Contenedor demo — puerto 8501**

Interfaz Streamlit de Sergio. Corre sobre `python:3.11-slim` sin imagen personalizada. Al levantar instala `streamlit`, `pandas` y `requests`, escribe la URL del backend en un archivo `.backend_url`, y arranca la app. La URL se lee desde archivo porque Streamlit cachea las variables de entorno al importar el módulo — el archivo sí se lee en cada llamada.

Desde dentro de la red de Docker Compose, el frontend resuelve el backend como `http://api:8000` usando el DNS interno de Docker.

---

## Modelo comercial vía API: Gemini 2.5 Flash

El modelo comercial es **Gemini 2.5 Flash** de Google (no GPT como quedó en el informe de avance). Tiene tier gratuito, ya teníamos la API key, y soporta `thinking_budget=0` para desactivar razonamiento extendido, lo que reduce latencia en clasificación.

Para T1 se usa **few-shot con majority voting**: 8 ejemplos en el prompt (uno por etiqueta), 3 llamadas con temperatura 0.5, se elige el label más votado. Empates se resuelven por confianza acumulada. Esta es la misma configuración del notebook A6 que produjo las métricas reportadas.

Para T2 es **zero-shot determinista**: temperatura 0, una sola llamada. Igual que el notebook A7.

Los prompts y parámetros están centralizados en `api/gemini_config.py` — si hay que cambiar algo, es un solo archivo.

---

## Estado actual de los modelos

| Slot | T1 | T2 | Estado |
|------|----|----|--------|
| LLM comercial API | Gemini 2.5 Flash | Gemini 2.5 Flash | Funcionando |
| Encoder fine-tuned | Modelo de Sergio (en `models/task1_encoder/`) | Pendiente | T1 cargando, T2 sin artefacto |
| Open-weight decoder | Ollama | Ollama | Pendiente — t3.small no tiene RAM para LLaMA 3 8B |

---

## Repositorio

`.gitignore` excluye `*.safetensors`, `*.bin` y `*.pt`. Los pesos se transfieren al EC2 vía SCP y quedan en `models/task1_encoder/` y `models/task2_encoder/`.

```
api/
  main.py           — endpoints FastAPI
  inference.py      — lógica de predicción (Gemini, encoder, Ollama)
  gemini_config.py  — prompts y parámetros de Gemini (fuente única)
  model_loader.py   — carga lazy de modelos HuggingFace, patrón singleton
  schemas.py        — contrato de datos con Pydantic

demo/
  app.py                 — interfaz Streamlit (Sergio)
  backend_production.py  — proxy HTTP al backend real

models/
  task1_encoder/    — pesos del encoder T1 (no en git, SCP al EC2)
  task2_encoder/    — pesos del encoder T2 (pendiente)

docker/
  Dockerfile
  docker-compose.yml

.env                — API keys (no en git)
```

---

## Flujo de una request

1. Usuario escribe texto en Streamlit (puerto 8501)
2. `backend_production.py` hace `POST /analyze` a `http://api:8000`
3. FastAPI parte el texto en párrafos — primero intenta separar por doble salto de línea; si viene todo junto, divide en ~5 chunks de oraciones
4. Por cada párrafo: T1 clasifica la función retórica (8 clases) y T2 detecta si hay contribución científica explícita, con el modelo que seleccionó el usuario
5. Devuelve JSON con segmentos, modelos usados y resumen
6. Streamlit renderiza tarjetas con color por etiqueta retórica

La pestaña de comparación fija T1 y varía los tres modelos de T2 sobre el mismo input.